In [1]:
import os, pprint

# Reliable HF download behavior in notebooks (set inside kernel)
os.environ["HF_HUB_READ_TIMEOUT"] = "120"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"   # enable faster transfer path
os.environ["HF_HUB_DOWNLOAD_RETRY"] = "10"
os.environ["EXECUTORCH_LOG_LEVEL"] = "debug"
os.environ["TRANSFORMERS_VERBOSITY"] = "info"
os.environ["HF_HUB_VERBOSITY"] = "debug"

# Quick check of envvars from inside the kernel
vars_to_check = [
    "HF_HUB_READ_TIMEOUT",
    "HF_HUB_ENABLE_HF_TRANSFER",
    "HF_HUB_DOWNLOAD_RETRY",
    "EXECUTORCH_LOG_LEVEL",
    "TRANSFORMERS_VERBOSITY",
    "HF_HUB_VERBOSITY",
]
print("Environment (checked from Python):")
pprint.pprint({v: os.environ.get(v) for v in vars_to_check})


Environment (checked from Python):
{'EXECUTORCH_LOG_LEVEL': 'debug',
 'HF_HUB_DOWNLOAD_RETRY': '10',
 'HF_HUB_ENABLE_HF_TRANSFER': '1',
 'HF_HUB_READ_TIMEOUT': '120',
 'HF_HUB_VERBOSITY': 'debug',
 'TRANSFORMERS_VERBOSITY': 'info'}


In [2]:
from transformers import logging
from huggingface_hub.utils import logging as hf_logging

# Transformers verbosity (info -> debug if you want even more)
logging.set_verbosity_info()      # or set_verbosity_debug() if you like noise
logging.enable_default_handler()
logging.enable_explicit_format()

# HF hub HTTP-level logs
hf_logging.set_verbosity_debug()


/Users/naren_work/Code/et_coreml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import os
from safetensors.torch import save_file

MODEL_ID = "Qwen/Qwen3-1.7B"   # change if you have a local mirror
OUT_DIR = "qwen3_fp16"

os.makedirs(OUT_DIR, exist_ok=True)

print("Loading model (this may take a while; verbose logs above will show progress)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,            # use dtype= (not torch_dtype) in newer transformers
    device_map="cpu",
    trust_remote_code=True,         # required for Qwen-3
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

# Save tokenizer and HF config (these calls will write config.json etc)
tokenizer.save_pretrained(OUT_DIR)
model.save_pretrained(OUT_DIR)  # Transformers will save safetensors if installed

Request 934b4f45-fb8a-4a95-9bbb-17482fe889a2: HEAD https://huggingface.co/Qwen/Qwen3-1.7B/resolve/main/config.json (authenticated: True)


Loading model (this may take a while; verbose logs above will show progress)...


Request 6e47b3b2-6ec3-46b1-aa33-640b78c4d779: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-1.7B/70d244cc86ccca08cf5af4e1e306ecf908b1ad5e/config.json (authenticated: True)
[INFO|configuration_utils.py:765] 2026-02-07 13:22:29,255 >> loading configuration file config.json from cache at /Users/naren_work/.cache/huggingface/hub/models--Qwen--Qwen3-1.7B/snapshots/70d244cc86ccca08cf5af4e1e306ecf908b1ad5e/config.json
[INFO|configuration_utils.py:839] 2026-02-07 13:22:29,257 >> Model config Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "float16",
  "eos_token_id": 151645,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 6144,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    

In [ ]:
# INSERT THIS CELL here (after saving the FP16 HF checkpoint, before convert_weights/export_llama)
# Minimal, official PT2E int4 quantization (weights-only, CoreML-safe)

from torch.ao.quantization.quantize_pt2e import prepare_pt2e, convert_pt2e
from torch.ao.quantization.quantizer.xnnpack_quantizer import (
    XNNPACKQuantizer,
    get_symmetric_quantization_config,
)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# If 'model' and 'tokenizer' are already in memory use them.
# Otherwise reload cleanly from the HF folder you saved earlier:
MODEL_DIR = "qwen3_fp16"
try:
    model  # if already defined in notebook
except NameError:
    model = AutoModelForCausalLM.from_pretrained(MODEL_DIR, dtype=torch.float16, device_map="cpu", trust_remote_code=True)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)

model.eval()

# Build the official PT2E quantizer (XNNPACKQuantizer)
quantizer = XNNPACKQuantizer()
quantizer.set_global(
    get_symmetric_quantization_config(
        is_per_channel=True,
    )
)

# Prepare for PT2E - official API
prepared = prepare_pt2e(model, quantizer)

# Calibration: use representative inputs if available; otherwise a few dummy batches
def calibrate(mod, tokenizer, batches=8, seq_len=64):
    for _ in range(batches):
        ids = torch.randint(0, tokenizer.vocab_size, (1, seq_len), dtype=torch.long)
        with torch.no_grad():
            mod(ids)

# Run calibration (tune batches/seq_len to your representative data if available)
calibrate(prepared, tokenizer, batches=20, seq_len=128)

# Convert to quantized model (official API)
quantized_model = convert_pt2e(prepared)
quantized_model.eval()

# Save quantized HF checkpoint in the standard HF layout (so ExecuTorch converter can read it)
quantized_model.save_pretrained("qwen3_1_7B_int8")
tokenizer.save_pretrained("qwen3_1_7B_int8")

print("Saved PT2E INT8 HF checkpoint in qwen3_1_7B_int8/ — ready for ExecuTorch conversion.")

/var/folders/ct/rv3z52fx5298nlp84rc1vf1c0000gq/T/ipykernel_65027/1263031661.py:24: DeprecationWarning: XNNPACKQuantizer is deprecated! Please use xnnpack quantizer in ExecuTorch (https://github.com/pytorch/executorch/tree/main/backends/xnnpack/quantizer) instead.
  quantizer = XNNPACKQuantizer()


TypeError: get_symmetric_quantization_config() got an unexpected keyword argument 'weight_qdtype'

In [5]:
from safetensors.torch import load_file
import torch

state = load_file("qwen3_fp16/model.safetensors")
torch.save(state, "qwen3_fp16/pytorch_model.bin", _use_new_zipfile_serialization=True)

In [6]:
import safetensors.torch as st

state = st.load_file("qwen3_fp16/model.safetensors")
[k for k in state if "absmax" in k or "scale" in k]


[]

In [10]:
!curl -L \
  "https://raw.githubusercontent.com/pytorch/executorch/main/examples/models/qwen3/config/1_7b_config.json" \
  -o qwen3_fp16/params.json

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   348  100   348    0     0    771      0 --:--:-- --:--:-- --:--:--   769


In [ ]:
!python -m executorch.examples.models.qwen3.convert_weights \
  qwen3_fp16 \
  ./qwen3_fp16/qwen3_fp16_et

  ### Just order matter here for passing arguments. Do not use argument names prefixed by --

Skipping import of cpp extensions due to incompatible torch version 2.9.1 for torchao version 0.14.0         Please see GitHub issue #2919 for more info
W0207 13:48:47.574000 85403 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
I tokenizers:regex.cpp:27] Registering override fallback regex
<frozen runpy>:128: RuntimeWarning: 'executorch.examples.models.qwen3.convert_weights' found in sys.modules after import of package 'executorch.examples.models.qwen3', but prior to execution of 'executorch.examples.models.qwen3.convert_weights'; this may result in unpredictable behaviour
Loading checkpoint...
Loading checkpoint from pytorch_model directory
Converting checkpoint...
Saving checkpoint...
Done.


In [17]:
!python -m executorch.tools.inspect ./qwen3_fp16/qwen3_fp16_et


/Users/naren_work/Code/et_coreml/bin/python: Error while finding module specification for 'executorch.tools.inspect' (ModuleNotFoundError: No module named 'executorch.tools')


In [ ]:
!python -m executorch.examples.models.llama.export_llama \
  --model qwen3_1_7b \
  --checkpoint qwen3_fp16/qwen3_fp16_et \
  --params qwen3_fp16/params.json \
  --output_name qwen3_fp16/qwen3_fp16_et_model.pte \
  --dtype fp16 \
  --use_kv_cache \
  --use_sdpa_with_kv_cache \
  --disable_dynamic_shape \
  --max_context_length 1024 \
  --max_seq_length 1024


Skipping import of cpp extensions due to incompatible torch version 2.9.1 for torchao version 0.14.0         Please see GitHub issue #2919 for more info
W0207 13:56:48.278000 90983 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
I tokenizers:regex.cpp:27] Registering override fallback regex
[INFO 2026-02-07 13:56:50,092 export_llama_lib.py:781] Applying quantizers: []
[INFO 2026-02-07 13:56:50,262 export_llama_lib.py:703] Checkpoint dtype: torch.float16
[INFO 2026-02-07 13:56:50,262 custom_kv_cache.py:367] Replacing KVCache with CustomKVCache. This modifies the model in place.
[INFO 2026-02-07 13:56:50,290 custom_ops.py:34] Looking for libcustom_ops_aot_lib.so in /Users/naren_work/Code/et_coreml/lib/python3.11/site-packages/executorch/extension/llm/custom_ops
[INFO 2026-02-07 13:56:50,291 custom_ops.py:39] Loading custom ops library: /Users/naren_work/Code/et_coreml/lib/python3.11/site-packages/executorch/exten

In [20]:
import torch
from executorch import exir
from executorch.exir import to_edge

from torch.ao.quantization.quantize_pt2e import prepare_pt2e, convert_pt2e
from torch.ao.quantization.quantizer.xnnpack_quantizer import (
    XNNPACKQuantizer,
    get_symmetric_quantization_config,
)

# Load FP16 ExecuTorch program
edge = exir.load("qwen3_fp16/qwen3_fp16_et_model.pte")
ep = edge.exported_program()
model = ep.module()
model.eval()

BadZipFile: File is not a zip file